In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install rank_bm25
import sys, re, os, time, numpy as np, pandas as pd
sys.path.insert(0, "/content/drive/MyDrive/esci/code")
from metrics import evaluate
from cross_encoder import product_to_text, esci_row_to_product
from rank_bm25 import BM25Okapi

BASE = "/content/drive/MyDrive/esci"
df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")
corpus = pd.read_parquet(f"{BASE}/dataset_v2.3/esci_electronics_v2.3_corpus.parquet")
val = df[df["part"] == "val"].copy()
results = []

# ---- Baseline 0: random order (the floor)
val["score"] = np.random.default_rng(1).random(len(val))
m, _ = evaluate(val); results.append({"model": "random", "format": "-", **m}); print("random:", m)

# ---- Baseline 1: BM25 (word statistics from the whole 51,318-product corpus)
tokenize = lambda s: re.findall(r"[a-z0-9]+", str(s).lower())
t0 = time.time()
corpus_text = [product_to_text({"title": t, "brand": b, "color": c}, "full")
               for t, b, c in zip(corpus["product_title"], corpus["product_brand"], corpus["product_color"])]
bm25 = BM25Okapi([tokenize(t) for t in corpus_text])
pid2idx = {p: i for i, p in enumerate(corpus["product_id"])}

for qid, idx in val.groupby("query_id").groups.items():
    q = val.loc[idx[0], "query"]
    ids = [pid2idx[p] for p in val.loc[idx, "product_id"]]
    val.loc[idx, "score"] = bm25.get_batch_scores(tokenize(q), ids)

m, per_q_bm25 = evaluate(val)
results.append({"model": "BM25", "format": "full", **m})
print("BM25  :", m, f"({time.time()-t0:.0f}s)")

# ---- save
os.makedirs(f"{BASE}/results", exist_ok=True)
pd.DataFrame(results).to_csv(f"{BASE}/results/baselines_val.csv", index=False)
per_q_bm25.to_csv(f"{BASE}/results/per_query_val_BM25.csv", index=False)
pd.DataFrame(results)

Mounted at /content/drive
random: {'ndcg@10': 0.7682, 'mrr': 0.7759, 'p@5': 0.6838}
BM25  : {'ndcg@10': 0.8722, 'mrr': 0.8832, 'p@5': 0.7924} (4s)


,model,format,ndcg@10,mrr,p@5
0,random,-,0.7682,0.7759,0.6838
1,BM25,full,0.8722,0.8832,0.7924


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os, time, torch, numpy as np, pandas as pd
sys.path.insert(0, "/content/drive/MyDrive/esci/code")
from metrics import evaluate
from cross_encoder import product_to_text, esci_row_to_product

BASE = "/content/drive/MyDrive/esci"
df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")
val = df[df["part"] == "val"].copy()
print("GPU:", torch.cuda.get_device_name(0), "| val rows:", len(val), "| val queries:", val["query_id"].nunique())

# `small` comes from the queries file (one value per query); `small_version` from the judgments file (per row)
chk = df.groupby("query_id").agg(small=("small", "first"), sv_max=("small_version", "max"), sv_min=("small_version", "min"))
print("small == small_version for every query:", bool((chk["small"] == chk["sv_max"]).all()),
      "| small_version same within each query:", bool((chk["sv_max"] == chk["sv_min"]).all()))

# product text in all 4 styles, made once
STYLES = ["title", "title_brand", "full", "full_bullets"]
MAXLEN = {"title": 128, "title_brand": 128, "full": 128, "full_bullets": 256}
prods = [esci_row_to_product(r) for _, r in val.iterrows()]
for s in STYLES:
    val[f"text_{s}"] = [product_to_text(p, s) for p in prods]
results = pd.read_csv(f"{BASE}/results/baselines_val.csv").to_dict("records")   # keeps random + BM25
print(pd.DataFrame(results))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4 | val rows: 11741 | val queries: 656
small == small_version for every query: True | small_version same within each query: False
    model format  ndcg@10     mrr     p@5
0  random      -   0.7682  0.7759  0.6838
1    BM25   full   0.8722  0.8832  0.7924


In [2]:
# how many queries / rows have mixed small_version?
mixed = chk[chk["sv_max"] != chk["sv_min"]]
print("queries with mixed small_version:", len(mixed), "of", len(chk))
print(df.loc[df["query_id"].isin(mixed.index)].groupby("split")["query_id"].nunique())

t = df[(df["split"] == "test") & (df["small"] == 1)]
print("\nTEST, small queries (507):")
print("  all rows of these queries     :", len(t))
print("  only rows with small_version=1:", int((t["small_version"] == 1).sum()))
print("  label mix, all rows   :", t["esci_label"].value_counts(normalize=True).round(3).to_dict())
print("  label mix, sv=1 rows  :", t.loc[t.small_version==1, "esci_label"].value_counts(normalize=True).round(3).to_dict())

queries with mixed small_version: 1 of 5665
split
train    1
Name: query_id, dtype: int64

TEST, small queries (507):
  all rows of these queries     : 9026
  only rows with small_version=1: 9026
  label mix, all rows   : {'S': 0.408, 'E': 0.399, 'I': 0.103, 'C': 0.09}
  label mix, sv=1 rows  : {'S': 0.408, 'E': 0.399, 'I': 0.103, 'C': 0.09}


In [3]:
from sentence_transformers import SentenceTransformer
t0 = time.time()
bi = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
q_emb = bi.encode(val["query"].tolist(), batch_size=256, normalize_embeddings=True, convert_to_numpy=True)
p_emb = bi.encode(val["text_full"].tolist(), batch_size=256, normalize_embeddings=True, convert_to_numpy=True)
val["score"] = (q_emb * p_emb).sum(axis=1)                 # cosine (vectors are normalised)
m, per_q = evaluate(val)
secs = round(time.time() - t0, 1)
results.append({"model": "SBERT all-MiniLM-L6-v2 (bi-encoder)", "format": "full", **m, "time_s": secs})
per_q.to_csv(f"{BASE}/results/per_query_val_SBERT_full.csv", index=False)
print("SBERT:", m, f"({secs}s)")
del bi; torch.cuda.empty_cache()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT: {'ndcg@10': 0.883, 'mrr': 0.9025, 'p@5': 0.8012} (17.8s)


In [4]:
from sentence_transformers import CrossEncoder

def load_ce(name, L):
    try:
        return CrossEncoder(name, max_length=L, device="cuda")
    except TypeError:                       # in case the library renamed the argument
        m = CrossEncoder(name, device="cuda"); m.max_length = L; return m

for name in ["cross-encoder/ms-marco-MiniLM-L-6-v2", "cross-encoder/ms-marco-MiniLM-L-12-v2"]:
    short = name.split("/")[-1]
    for s in STYLES:
        t0 = time.time()
        ce = load_ce(name, MAXLEN[s])
        pairs = list(zip(val["query"], val[f"text_{s}"]))
        val["score"] = ce.predict(pairs, batch_size=128, show_progress_bar=False)
        m, per_q = evaluate(val)
        secs = round(time.time() - t0, 1)
        results.append({"model": f"{short} (zero-shot)", "format": s, **m, "time_s": secs})
        per_q.to_csv(f"{BASE}/results/per_query_val_{short}_zeroshot_{s}.csv", index=False)
        print(f"{short:28s} {s:13s} {m}  ({secs}s)")
        del ce; torch.cuda.empty_cache()

table = pd.DataFrame(results)
table.to_csv(f"{BASE}/results/baselines_val.csv", index=False)
table.sort_values("ndcg@10", ascending=False)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ms-marco-MiniLM-L-12-v2      full          {'ndcg@10': 0.9134, 'mrr': 0.9346, 'p@5': 0.8299}  (19.3s)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ms-marco-MiniLM-L-12-v2      full_bullets  {'ndcg@10': 0.9144, 'mrr': 0.932, 'p@5': 0.8284}  (39.7s)


,model,format,ndcg@10,mrr,p@5,time_s
10,ms-marco-MiniLM-L-12-v2 (zero-shot),full_bullets,0.9144,0.9320,0.8284,39.7
8,ms-marco-MiniLM-L-12-v2 (zero-shot),title_brand,0.9139,0.9379,0.8302,18.2
9,ms-marco-MiniLM-L-12-v2 (zero-shot),full,0.9134,0.9346,0.8299,19.3
7,ms-marco-MiniLM-L-12-v2 (zero-shot),title,0.9134,0.9384,0.8293,21.1
6,ms-marco-MiniLM-L-6-v2 (zero-shot),full_bullets,0.9128,0.9305,0.8299,17.6
5,ms-marco-MiniLM-L-6-v2 (zero-shot),full,0.9121,0.9348,0.8302,9.8
4,ms-marco-MiniLM-L-6-v2 (zero-shot),title_brand,0.9117,0.9323,0.8271,9.3
3,ms-marco-MiniLM-L-6-v2 (zero-shot),title,0.9096,0.9338,0.8280,12.8
2,SBERT all-MiniLM-L6-v2 (bi-encoder),full,0.8830,0.9025,0.8012,17.8
1,BM25,full,0.8722,0.8832,0.7924,NaN
